# Classificação de Imagens com MobileNetV3 (timm)

Este notebook utiliza a biblioteca `timm` para classificar as imagens contidas na pasta `imagens` usando o modelo `mobilenetv3_small_100.lamb_in1k`.

In [ ]:
import torch
import timm
from PIL import Image
import os
import requests
import matplotlib.pyplot as plt
import numpy as np
from tqdm.notebook import tqdm

print(f"Torch version: {torch.__version__}")
print(f"Timm version: {timm.__version__}")

## 1. Carregar o Modelo e Configurações

In [ ]:
model_name = 'timm/mobilenetv3_small_100.lamb_in1k'
print(f"📦 Carregando modelo: {model_name}...")
model = timm.create_model(model_name, pretrained=True)
model.eval()

device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)

# Resolver configurações de dados (resolução, normalização, etc)
data_config = timm.data.resolve_model_data_config(model)
transform = timm.data.create_transform(**data_config, is_training=False)

print(f"✅ Modelo carregado no dispositivo: {device}")

## 2. Buscar Labels do ImageNet

In [ ]:
try:
    labels_url = "https://raw.githubusercontent.com/pytorch/hub/master/imagenet_classes.txt"
    labels = requests.get(labels_url).text.splitlines()
    print(f"Total de labels carregados: {len(labels)}")
except Exception as e:
    print(f"Erro ao baixar labels: {e}. Usando labels genéricos.")
    labels = [f"Classe {i}" for i in range(1000)]

## 3. Função de Classificação e Visualização

In [ ]:
def classify_and_show(img_path):
    img = Image.open(img_path).convert('RGB')
    input_tensor = transform(img).unsqueeze(0).to(device)
    
    with torch.no_grad():
        output = model(input_tensor)
        
    probabilities = torch.nn.functional.softmax(output[0], dim=0)
    top5_prob, top5_catid = torch.topk(probabilities, 5)
    
    # Visualização Premium
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4), gridspec_kw={'width_ratios': [1, 1.5]})
    
    # Imagem
    ax1.imshow(img)
    ax1.set_title(os.path.basename(img_path))
    ax1.axis('off')
    
    # Gráfico de barras com as predições
    y_pos = np.arange(5)
    probs = top5_prob.cpu().numpy() * 100
    names = [labels[idx] for idx in top5_catid.cpu().numpy()]
    
    bars = ax2.barh(y_pos, probs, color='skyblue')
    ax2.set_yticks(y_pos)
    ax2.set_yticklabels(names)
    ax2.invert_yaxis()  # Maior probabilidade no topo
    ax2.set_xlabel('Confiança (%)')
    ax2.set_xlim(0, 100)
    
    # Adicionar porcentagem no final das barras
    for i, v in enumerate(probs):
        ax2.text(v + 1, i, f'{v:.1f}%', color='blue', fontweight='bold', va='center')
        
    plt.tight_layout()
    plt.show()

## 4. Executando na Pasta 'imagens'

In [ ]:
img_dir = "imagens"
valid_extensions = (".png", ".jpg", ".jpeg")

if os.path.exists(img_dir):
    images = sorted([f for f in os.listdir(img_dir) if f.lower().endswith(valid_extensions)])
    
    if not images:
        print(f"Nenhuma imagem encontrada em '{img_dir}'.")
    else:
        print(f"🚀 Iniciando classificação em {len(images)} imagens...")
        for img_name in images:
            img_path = os.path.join(img_dir, img_name)
            classify_and_show(img_path)
else:
    print(f"❌ Pasta '{img_dir}' não encontrada.")